# Sweep Runner

Select an experiment, check W&B for completed runs, and launch remaining hyperparameter combinations.

In [1]:
import os
import glob
import ipywidgets as widgets
import itertools
import subprocess
from pathlib import Path

import wandb
import yaml
from omegaconf import OmegaConf

REPO_ROOT = Path(os.path.abspath("")).parent
CONF_DIR = REPO_ROOT / "JacobianODE" / "jacobians" / "conf"
EXP_DIR = CONF_DIR / "experiment"

## 1. Select an experiment

In [6]:
# Discover experiment YAML files (exclude old/ subdirectory)
experiment_files = sorted(
    p for p in EXP_DIR.glob("*.yaml")
    if p.is_file()
)

experiments = {p.stem: p for p in experiment_files}
for i, name in enumerate(experiments):
    print(f"  [{i}] {name}")

dropdown = widgets.Dropdown(
    options=list(experiments.keys()),
    description="Experiment:",
    value=list(experiments.keys())[0],
    layout=widgets.Layout(width="750px")
)
display(dropdown)

# Then read dropdown.value whenever you need it

  [0] lorenz_partial_spline_coupling_geometric_noise_sweep
  [1] lorenz_partial_spline_coupling_geometric_noise_sweep_many_step
  [2] lorenz_spline_coupling_geometric_noise_sweep
  [3] wmtask_spline_coupling_geometric_noise_sweep
  [4] wmtask_spline_coupling_sweep


Dropdown(description='Experiment:', layout=Layout(width='750px'), options=('lorenz_partial_spline_coupling_geo…

In [7]:
# ── Pick one ──────────────────────────────────────────────────────────────────
# EXPERIMENT = list(experiments.keys())[0]  # <-- change index or set name directly
EXPERIMENT = dropdown.value

print(f"Selected: {EXPERIMENT}")

Selected: lorenz_partial_spline_coupling_geometric_noise_sweep_many_step


## 2. Parse experiment config (sweep grid, W&B project/group)

In [21]:
# Load experiment YAML and the base configs it depends on to resolve interpolations
exp_path = experiments[EXPERIMENT]
exp_cfg = OmegaConf.load(exp_path)

# Load base configs for variable resolution
data_name = None
sweeper_name = None
for d in OmegaConf.to_container(exp_cfg.get("defaults", []), resolve=False) or []:
    if isinstance(d, dict):
        if "override /data" in d:
            data_name = d["override /data"]
        if "override /hydra/sweeper" in d:
            sweeper_name = d["override /hydra/sweeper"]

if data_name:
    data_cfg = OmegaConf.load(CONF_DIR / "data" / f"{data_name}.yaml")
else:
    data_cfg = OmegaConf.create()

model_name = None
for d in OmegaConf.to_container(exp_cfg.get("defaults", []), resolve=False) or []:
    if isinstance(d, dict) and "override /model" in d:
        model_name = d["override /model"]
if model_name:
    model_cfg = OmegaConf.load(CONF_DIR / "model" / f"{model_name}.yaml")
else:
    model_cfg = OmegaConf.create()

training_cfg = OmegaConf.load(CONF_DIR / "training" / "training.yaml")

# Merge: base < data < model < training < experiment (experiment wins)
# Exclude the hydra block — it contains ${now:...} which is a Hydra-only
# resolver and will fail under plain OmegaConf.resolve().
exp_cfg_no_hydra = {k: v for k, v in OmegaConf.to_container(exp_cfg).items()
                    if k not in ("defaults", "hydra")}

base_cfg = OmegaConf.load(CONF_DIR / "config.yaml")
base_no_hydra = {k: v for k, v in OmegaConf.to_container(base_cfg).items()
                 if k != "hydra"}

merged = OmegaConf.merge(
    base_no_hydra,
    {"data": data_cfg},
    {"model": model_cfg},
    {"training": training_cfg},
    exp_cfg_no_hydra,
)
OmegaConf.resolve(merged)

# ── Extract W&B coordinates ──────────────────────────────────────────────────
WANDB_ENTITY = merged.get("wandb_entity", "JacobianODE")
WANDB_PROJECT = merged.get("wandb_project", None)
WANDB_GROUP = merged.get("wandb_group", None)

# ── Extract sweep grid from the sweeper config ──────────────────────────────
assert sweeper_name is not None, (
    f"Experiment {EXPERIMENT} has no 'override /hydra/sweeper' in defaults. "
    f"Add one (e.g. grid_lc_kl) to define the sweep grid."
)
sweeper_cfg = OmegaConf.load(CONF_DIR / "hydra" / "sweeper" / f"{sweeper_name}.yaml")
sweep_params_raw = OmegaConf.to_container(sweeper_cfg.params, resolve=False)

# Parse each param: "0,1e-6,..." -> list of float values
sweep_params = {}
for key, val_str in sweep_params_raw.items():
    values = [float(v) for v in val_str.split(",")]
    sweep_params[key] = values

# Full grid = cartesian product
param_names = list(sweep_params.keys())
param_values = list(sweep_params.values())
full_grid = [dict(zip(param_names, combo)) for combo in itertools.product(*param_values)]

print(f"W&B entity:  {WANDB_ENTITY}")
print(f"W&B project: {WANDB_PROJECT}")
print(f"W&B group:   {WANDB_GROUP}")
print(f"Sweeper:     {sweeper_name}")
print(f"Sweep params: {param_names}")
print(f"Grid sizes:   {[len(v) for v in param_values]}")
print(f"Total combinations: {len(full_grid)}")

W&B entity:  JacobianODE
W&B project: Lorenz_INDall_N25_D1_NormTrue_T3__JacobianODE
W&B group:   spline_coupling__geometric_noise__sweep_lc_x_kl_dyn_100step
Sweeper:     grid_lc_kl
Sweep params: ['training.lightning.loop_closure_weight', 'training.lightning.kl_dyn_weight']
Grid sizes:   [9, 9]
Total combinations: 81


## 3. Query W&B for completed runs

In [22]:
from JacobianODE.jacobians.tuning.run_status import (
    meets_early_stopping_criterion,
    hit_slurm_walltime,
    is_run_effectively_done,
)
import math


def _get_config_value(run_config, dotted_key):
    """Extract a value from a W&B run config using a dotted key like
    'training.lightning.loop_closure_weight'. Handles both nested dicts
    and flat dotted keys."""
    parts = dotted_key.split(".")
    obj = run_config
    for part in parts:
        if isinstance(obj, dict) and part in obj:
            obj = obj[part]
        else:
            obj = None
            break
    if obj is not None:
        return obj
    return run_config.get(dotted_key)

In [23]:
api = wandb.Api(timeout=90)
project_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

run_filters = {}
if WANDB_GROUP:
    run_filters["group"] = WANDB_GROUP

try:
    all_runs = api.runs(project_path, filters=run_filters if run_filters else None)
    print(f"Found {len(all_runs)} total runs in {project_path}")
    if WANDB_GROUP:
        print(f"  (filtered to group: {WANDB_GROUP})")
except Exception as e:
    all_runs = []
    print(f"Could not fetch runs: {e}")
    print("(Project may not exist yet — all combinations will be marked as remaining.)")

Found 1 total runs in JacobianODE/Lorenz_INDall_N25_D1_NormTrue_T3__JacobianODE
  (filtered to group: spline_coupling__geometric_noise__sweep_lc_x_kl_dyn_100step)


In [24]:
# ── Classify runs ─────────────────────────────────────────────────────────────
max_epochs = int(OmegaConf.select(merged, "training.trainer_params.max_epochs", default=1000))
es_patience = int(OmegaConf.select(merged, "training.early_stopping.early_stopping_patience", default=5))
es_pct_thresh = float(OmegaConf.select(merged, "training.early_stopping.percent_thresh", default=0.01))
es_min_epochs = int(OmegaConf.select(merged, "training.early_stopping.min_epochs", default=10))
es_monitor = OmegaConf.select(merged, "training.early_stopping.monitor", default="mean val loss")

# SLURM walltime from the slurm config used by this experiment
slurm_cfg_name = None
for d in OmegaConf.to_container(exp_cfg.get("defaults", []), resolve=False) or []:
    if isinstance(d, dict) and "override /slurm" in d:
        slurm_cfg_name = d["override /slurm"]
if slurm_cfg_name is None:
    for d in OmegaConf.to_container(OmegaConf.load(CONF_DIR / "config.yaml").get("defaults", []), resolve=False) or []:
        if isinstance(d, dict) and "slurm" in d:
            slurm_cfg_name = d["slurm"]
            break
        elif isinstance(d, str) and d.startswith("slurm:"):
            slurm_cfg_name = d.split(":")[1].strip()
            break

slurm_timeout_min = None
if slurm_cfg_name and slurm_cfg_name != "none":
    slurm_cfg_path = CONF_DIR / "slurm" / f"{slurm_cfg_name}.yaml"
    if slurm_cfg_path.exists():
        slurm_cfg = OmegaConf.load(slurm_cfg_path)
        slurm_timeout_min = OmegaConf.select(slurm_cfg, "hydra.launcher.timeout_min", default=None)

WALLTIME_TOLERANCE_MIN = 5
print(f"SLURM timeout: {slurm_timeout_min} min" + (f"  (tolerance: {WALLTIME_TOLERANCE_MIN} min)" if slurm_timeout_min else " (not set)"))

finished_configs = []   # list of param dicts that are done
running_configs = []    # currently running
failed_configs = []     # crashed / failed and NOT converged

n_finished = 0
n_early_stopped = 0
n_running = 0
n_failed = 0
n_converged_crashed = 0
n_walltime_killed = 0

for run in all_runs:
    # Extract the swept param values from this run's config
    run_params = {}
    skip = False
    for pname in param_names:
        val = _get_config_value(run.config, pname)
        if val is None:
            skip = True
            break
        run_params[pname] = float(val)
    if skip:
        continue

    if run.state == "finished":
        finished_configs.append(run_params)
        n_finished += 1
        last_epoch = run.summary.get("epoch", run.summary.get("trainer/current_epoch"))
        if last_epoch is not None and int(last_epoch) < max_epochs - 1:
            n_early_stopped += 1

    elif run.state == "running":
        running_configs.append(run_params)
        n_running += 1

    elif run.state in ("crashed", "failed"):
        parts = [f"{k.split('.')[-1]}={run_params[k]}" for k in param_names]

        # Use the shared is_run_effectively_done (checks convergence + walltime)
        if is_run_effectively_done(
            run,
            monitor=es_monitor,
            patience=es_patience,
            percent_thresh=es_pct_thresh,
            min_epochs=es_min_epochs,
            slurm_timeout_min=slurm_timeout_min,
            walltime_tolerance_min=WALLTIME_TOLERANCE_MIN,
        ):
            # Determine reason for logging
            converged, last_ep = meets_early_stopping_criterion(
                run, monitor=es_monitor, patience=es_patience,
                percent_thresh=es_pct_thresh, min_epochs=es_min_epochs,
            )
            if converged:
                reason = f"converged (epoch {last_ep})"
                n_converged_crashed += 1
            else:
                reason = "hit SLURM walltime"
                n_walltime_killed += 1
            finished_configs.append(run_params)
            print(f"  {run.id}: {reason} — {', '.join(parts)}")
        else:
            failed_configs.append(run_params)
            n_failed += 1

print(f"\nFinished:              {n_finished}  ({n_early_stopped} early-stopped)")
print(f"Crashed but converged: {n_converged_crashed}  (counted as done)")
print(f"Hit SLURM walltime:    {n_walltime_killed}  (counted as done)")
print(f"Running:               {n_running}")
print(f"Crashed/Failed:        {n_failed}  (need re-run)")

SLURM timeout: 180 min  (tolerance: 5 min)

Finished:              0  (0 early-stopped)
Crashed but converged: 0  (counted as done)
Hit SLURM walltime:    0  (counted as done)
Running:               0
Crashed/Failed:        1  (need re-run)


## 4. Compute remaining hyperparameter combinations

In [25]:
def params_match(a, b, tol=1e-12):
    """Check if two param dicts match (float comparison with tolerance)."""
    for key in a:
        va, vb = float(a[key]), float(b[key])
        if va == 0 and vb == 0:
            continue
        if va == 0 or vb == 0:
            return False
        if abs(va - vb) / max(abs(va), abs(vb)) > tol:
            return False
    return True


def is_done_or_running(combo, done_list, running_list):
    """Check if a grid combo is already finished or running."""
    for done in done_list:
        if params_match(combo, done):
            return True
    for running in running_list:
        if params_match(combo, running):
            return True
    return False


# ── INCLUDE_FAILED: set True to re-run failed jobs, False to skip them ────────
INCLUDE_FAILED = True

remaining = []
for combo in full_grid:
    if is_done_or_running(combo, finished_configs, running_configs):
        continue
    if not INCLUDE_FAILED and any(params_match(combo, f) for f in failed_configs):
        continue
    remaining.append(combo)

print(f"Total grid:     {len(full_grid)}")
print(f"Already done:   {len(finished_configs)}")
print(f"Running:        {len(running_configs)}")
print(f"Remaining:      {len(remaining)}")

if remaining:
    print(f"\nFirst 5 remaining combos:")
    for combo in remaining[:5]:
        parts = [f"{k.split('.')[-1]}={v}" for k, v in combo.items()]
        print(f"  {', '.join(parts)}")
    if len(remaining) > 5:
        print(f"  ... and {len(remaining) - 5} more")

Total grid:     81
Already done:   0
Running:        0
Remaining:      81

First 5 remaining combos:
  loop_closure_weight=0.0, kl_dyn_weight=0.0
  loop_closure_weight=0.0, kl_dyn_weight=1e-06
  loop_closure_weight=0.0, kl_dyn_weight=1e-05
  loop_closure_weight=0.0, kl_dyn_weight=0.0001
  loop_closure_weight=0.0, kl_dyn_weight=0.001
  ... and 76 more


## 5. Generate bash command

In [26]:
# ── OPTIONS ────────────────────────────────────────────────────────────────────
RUN_MODE = "remaining"  # "full" = run entire experiment, "remaining" = only missing combos
SLURM = True            # True = submit via SLURM (slurm=default), False = local (slurm=none)
DRY_RUN = True          # True = print command only, False = also execute it

ENV_PREFIX = "HYDRA_FULL_ERROR=1"


def _fmt_value(v):
    """Format a float for Hydra override (avoid scientific notation issues)."""
    if v == 0:
        return "0"
    if v == int(v) and abs(v) < 1e6:
        return str(int(v))
    return repr(v)


if RUN_MODE == "full":
    cmd = (
        f"cd {REPO_ROOT} && {ENV_PREFIX} "
        f"python -m JacobianODE.jacobians.run_jacobians --multirun "
        f"experiment={EXPERIMENT}"
    )
    if not SLURM:
        cmd += " slurm=none"
    print("Command (full sweep):")
    print(cmd)

elif RUN_MODE == "remaining":
    if not remaining:
        print("Nothing remaining to run!")
    else:
        # Check if remaining is a clean sub-grid (cartesian product)
        remaining_value_sets = {pname: set() for pname in param_names}
        for combo in remaining:
            for pname in param_names:
                remaining_value_sets[pname].add(combo[pname])
        remaining_as_subgrid = list(
            itertools.product(*[sorted(remaining_value_sets[p]) for p in param_names])
        )
        is_subgrid = len(remaining_as_subgrid) == len(remaining)

        if is_subgrid and len(remaining) > 1:
            # Clean sub-grid: use default Hydra sweeper with comma-separated values
            overrides = []
            for pname in param_names:
                vals = sorted(remaining_value_sets[pname])
                val_str = ",".join(_fmt_value(v) for v in vals)
                overrides.append(f"'{pname}={val_str}'")

            cmd = (
                f"cd {REPO_ROOT} && {ENV_PREFIX} "
                f"python -m JacobianODE.jacobians.run_jacobians --multirun "
                f"experiment={EXPERIMENT} "
                + " ".join(overrides)
            )
            if not SLURM:
                cmd += " slurm=none"
            print(f"Command (sub-grid, {len(remaining)} combos):")
            print(cmd)
        else:
            # Not a clean sub-grid: use our forked list sweeper (supports nested dicts).
            # Hydra's CLI nests dotted keys, but our plugin flattens them back.
            overrides = ["'hydra/sweeper=list'"]
            for pname in param_names:
                vals = [_fmt_value(combo[pname]) for combo in remaining]
                val_str = ",".join(vals)
                overrides.append(f"'+hydra.sweeper.list_params.{pname}=[{val_str}]'")

            cmd = (
                f"cd {REPO_ROOT} && {ENV_PREFIX} "
                f"python -m JacobianODE.jacobians.run_jacobians --multirun "
                f"experiment={EXPERIMENT} "
                + " ".join(overrides)
            )
            if not SLURM:
                cmd += " slurm=none"
            print(f"Command (list sweep, {len(remaining)} combos):")
            print(cmd)

Command (sub-grid, 81 combos):
cd /orcd/home/002/eisenaj/code/JacobianODE && HYDRA_FULL_ERROR=1 python -m JacobianODE.jacobians.run_jacobians --multirun experiment=lorenz_partial_spline_coupling_geometric_noise_sweep_many_step 'training.lightning.loop_closure_weight=0,1e-06,1e-05,0.0001,0.001,0.01,0.1,1,10' 'training.lightning.kl_dyn_weight=0,1e-06,1e-05,0.0001,0.001,0.01,0.1,1,10'


In [27]:
# ── Execute (only if DRY_RUN is False) ────────────────────────────────────────
if not DRY_RUN and RUN_MODE == "remaining" and not remaining:
    print("Nothing to run.")
elif not DRY_RUN:
    print(f"Launching...\n")
    result = subprocess.run(cmd, shell=True, capture_output=False)
    print(f"\nExit code: {result.returncode}")
else:
    print("\n(DRY_RUN=True — set to False and re-run this cell to execute)")


(DRY_RUN=True — set to False and re-run this cell to execute)


## 6. Run a single combination locally

In [28]:
# Build human-readable labels for every grid combo
_combo_labels = []
for i, combo in enumerate(full_grid):
    parts = [f"{k.split('.')[-1]}={_fmt_value(v)}" for k, v in combo.items()]
    _combo_labels.append(f"[{i}] {', '.join(parts)}")

single_run_dropdown = widgets.Dropdown(
    options=list(zip(_combo_labels, range(len(full_grid)))),
    description="Combo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)
display(single_run_dropdown)

Dropdown(description='Combo:', layout=Layout(width='80%'), options=(('[0] loop_closure_weight=0, kl_dyn_weight…

In [ ]:
SINGLE_DRY_RUN = False  # Set to False to execute

combo_idx = single_run_dropdown.value
chosen_combo = full_grid[combo_idx]

overrides = [f"{k}={_fmt_value(v)}" for k, v in chosen_combo.items()]

single_cmd = (
    f"cd {REPO_ROOT} && {ENV_PREFIX} "
    f"python -m JacobianODE.jacobians.run_jacobians "
    f"experiment={EXPERIMENT} "
    + " ".join(overrides)
    + " slurm=none"
)

print(f"Selected combo [{combo_idx}]:")
for k, v in chosen_combo.items():
    print(f"  {k} = {v}")
print(f"\nCommand:\n{single_cmd}")

if not SINGLE_DRY_RUN:
    print(f"\nLaunching locally...\n")
    result = subprocess.run(single_cmd, shell=True, capture_output=False)
    print(f"\nExit code: {result.returncode}")
else:
    print("\n(SINGLE_DRY_RUN=True — set to False and re-run this cell to execute)")

Selected combo [19]:
  training.lightning.loop_closure_weight = 1e-05
  training.lightning.kl_dyn_weight = 1e-06

Command:
cd /orcd/home/002/eisenaj/code/JacobianODE && HYDRA_FULL_ERROR=1 python -m JacobianODE.jacobians.run_jacobians experiment=lorenz_partial_spline_coupling_geometric_noise_sweep_many_step training.lightning.loop_closure_weight=1e-05 training.lightning.kl_dyn_weight=1e-06 slurm=none

Launching locally...

[2026-04-07 17:20:36,170][JacobianLogger][INFO] - Starting JacobianODE training
[2026-04-07 17:20:36,219][JacobianLogger][INFO] - Number of available GPUs: 1
[2026-04-07 17:20:36,232][JacobianLogger][INFO] - Configuration:
logger: wandb
wandb_entity: JacobianODE
wandb_project: Lorenz_INDall_N${data.train_test_params.delay_embedding_params.n_delays}_D${data.train_test_params.delay_embedding_params.delay_spacing}_Norm${data.postprocessing.normalize}_T${model.n_target_dims}__JacobianODE
wandb_group: spline_coupling__geometric_noise__sweep_lc_x_kl_dyn_100step
optuna_study

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/eisenaj/.netrc.


[2026-04-07 17:20:41,852][JacobianODE.jacobians.training.model_factory][INFO] - Created model with 7,161,561 parameters
[2026-04-07 17:20:41,853][JacobianLogger][INFO] - Number of training trajectory examples: 23.342k
[2026-04-07 17:20:41,853][JacobianLogger][INFO] - Number of training trajectory points: 25.872k
[2026-04-07 17:20:41,853][JacobianLogger][INFO] - Number of training data points: 646.800k
[2026-04-07 17:20:41,853][JacobianLogger][INFO] - Total number of model parameters: 7161.561k


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: Currently logged in as: adamjeisen (JacobianODE) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run ntpq1gjv
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/wandb/run-20260407_172042-ntpq1gjv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run MLP__hidden_dim_[256, 1024, 2048, 2048]__num_layers_4__residuals_False__dropout_0.0__activation_silu__batch_size_32__optimizer_AdamW__optimizer_kwargs_{'lr': 0.0001, 'weight_decay': 0.0001}__use_scheduler_True__scheduler_type_cosine__min_lr_1e-06__k_scale_1__gradient_clip_val_1.0__gradient_clip_algorithm_norm__alpha_teacher_forcing_1__teacher_forcing_annealing_True__gamma_teacher_forcing_0.999__teacher_forcing_update_interval_5__teacher_forcing_steps_1__min_alpha_teacher_forcing_0__alpha_vali

[2026-04-07 17:20:44,068][pytorch_lightning.utilities.rank_zero][INFO] - GPU available: True (cuda), used: True
[2026-04-07 17:20:44,069][pytorch_lightning.utilities.rank_zero][INFO] - TPU available: False, using: 0 TPU cores
[2026-04-07 17:20:44,069][pytorch_lightning.utilities.rank_zero][INFO] - 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
[2026-04-07 17:20:44,070][JacobianODE.jacobians.training.trainer][INFO] - Starting training run: MLP__hidden_dim_[256, 1024, 2048, 2048]__num_layers_4__residuals_False__dropout_0.0__activation_silu__batch_size_32__optimizer_AdamW__optimizer_kwargs_{'lr': 0.0001, 'weight_decay': 0.0001}__use_scheduler_True__scheduler_type_cosine__min_lr_1e-06__k_scale_1__gradient_clip_val_1.0__gradient_clip_algorithm_norm__alpha_teacher_forcing_1__teacher_forcing_annealing_True__gamma

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv 1gjv 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━━━━━━━━━━━━━━━ 200/200 0:00:18 • 0:00:00 10.75it/s v_num: 1gjv
Epoch 0/149 ━━━━━━